# 05 · The model ladder (Step 6)

**Goal:** compare every syllabus model on the full feature table (240 features), in one fair contest:
the same 5 folds, the same metrics, the same leak-free rules.

**Models:** logistic regression, WoE scorecard, Naive Bayes (raw and on PCA), decision tree, KNN,
linear SVM, RBF SVM (Nystroem), random forest, AdaBoost, HistGradientBoosting, XGBoost, EBM,
plus LightGBM v1 from Step 5 (loaded, not retrained).

**Time:** about 25–40 minutes in total; AdaBoost and the random forest are the slowest.
**You can stop and re-run at any time:** every finished model's predictions are saved in
`data/processed/oof/`, and re-running loads them instead of retraining.

**Saved results:**

- `reports/results_step6_ladder.csv`, `reports/scorecard_points.csv`
- `reports/figures/fig14_ladder_roc_pr.png`, `reports/figures/fig15_tree_depth_curve.png`
- MLflow (`make mlflow`)

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, roc_auc_score
from threadpoolctl import threadpool_limits

from creditrisk.config import PROJECT_ROOT, get_path, set_seed
from creditrisk.data.split import load_splits
from creditrisk.evaluation.bootstrap import paired_bootstrap
from creditrisk.evaluation.cv import load_result, results_table, run_or_load
from creditrisk.features.build import load_features
from creditrisk.models import baselines
from creditrisk.models.baselines import default_threads
from creditrisk.models.data import column_groups, get_xy
from creditrisk.models.ladder import ladder_models, random_forest, woe_scorecard
from creditrisk.models.transformers import scorecard_points

warnings.filterwarnings("ignore", message="Missing values detected")  # an EBM plotting notice
# HistGradientBoosting uses OpenMP threads: same CPU fix as LightGBM in Step 4
THREAD_LIMIT = threadpool_limits(limits=default_threads(), user_api="openmp")

SEED = set_seed()
FIG = get_path("figures")
facts = {}

features = load_features()
splits = load_splits()
X, y, folds, ids = get_xy(features, splits, "train")
groups = column_groups(X)
print(f"train split: {X.shape[0]:,} rows x {X.shape[1]} features | threads: {default_threads()}")

## Run the ladder

Each model prints one line per fold while it trains, then a summary line. Models that are
already saved are loaded instantly. To retrain one, use `force=True`.

Preprocessing and feature selection are fitted inside each training fold. KNN, the SVMs,
AdaBoost and EBM first keep their top features, chosen by a quick LightGBM on that fold only.

In [ ]:
results = {}
try:
    results["lightgbm_v1"] = load_result("step5_ablation_5", ids=ids, display_name="lightgbm_v1")
    print("lightgbm_v1: loaded from Step 5")
except FileNotFoundError:
    results["lightgbm_v1"] = run_or_load(lambda: baselines.lightgbm(seed=SEED), X, y, folds,
                                         "step6_lightgbm_v1", ids=ids, step=6, verbose=True)

for name, make_model in ladder_models(groups, seed=SEED).items():
    r = run_or_load(make_model, X, y, folds, f"step6_{name}", ids=ids, step=6, verbose=True)
    r.name = name
    results[name] = r
    s = r.summary
    print(f"{name:<24} ROC-AUC {s['roc_auc']:.4f} "
          f"[{s['roc_auc_low']:.4f}, {s['roc_auc_high']:.4f}]")

## Results table (out-of-fold, train split)

In [ ]:
table = results_table(results.values())
table.to_csv(PROJECT_ROOT / "reports" / "results_step6_ladder.csv")
display(table)
print("saved reports/results_step6_ladder.csv")

auc = table["ROC-AUC"]
facts["auc_by_model"] = auc.to_dict()
facts["best_model"] = (auc.index[0], float(auc.iloc[0]))
facts["runner_up"] = (auc.index[1], float(auc.iloc[1]))

## Are the gaps real? (paired bootstrap, same resamples for both models)
- best vs runner-up: is the winner really better?
- best vs logistic regression: how much does boosting add over a linear model?
- best vs WoE scorecard: the "price of transparency"

In [ ]:
best = auc.index[0]
pairs = [(best, auc.index[1]), (best, "logistic_regression"), (best, "woe_scorecard")]
rows = []
for a, b in pairs:
    d = paired_bootstrap(y, results[a].oof, results[b].oof, n_boot=300)
    rows.append({"comparison": f"{a} - {b}", "difference": round(d["difference"], 4),
                 "95% CI": f"{d['low']:+.4f} to {d['high']:+.4f}", "significant": d["significant"]})
comparisons = pd.DataFrame(rows).set_index("comparison")
facts["comparisons"] = comparisons["difference"].to_dict()
display(comparisons)

## ROC and precision-recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for name in table.index:
    RocCurveDisplay.from_predictions(y, results[name].oof, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y, results[name].oof, name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set_title("ROC curves (out-of-fold, train split)")
axes[1].axhline(y.mean(), color="grey", ls="--", lw=0.8)
axes[1].set_title("Precision-recall curves (dashed = random model)")
for ax in axes:
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "fig14_ladder_roc_pr.png", dpi=150, bbox_inches="tight")
print("saved reports/figures/fig14_ladder_roc_pr.png")

## Bias vs variance: the decision-tree depth curve

Trees of growing depth are trained on folds 2–5 (a 60,000-row sample, to keep it quick) and
checked on fold 1. The training score keeps rising, but the held-out score peaks and then falls.
That gap is overfitting (high variance); very shallow trees are too simple (high bias).

In [ ]:
fit_rows = np.flatnonzero(folds != 1)
rng = np.random.default_rng(SEED)
fit_rows = rng.choice(fit_rows, size=min(60_000, len(fit_rows)), replace=False)
check_rows = np.flatnonzero(folds == 1)
curve = []
for depth in [2, 4, 6, 8, 10, 14, 20]:
    tree = baselines.decision_tree(groups, max_depth=depth, min_samples_leaf=1, seed=SEED)
    tree.fit(X.iloc[fit_rows], y[fit_rows])
    curve.append({
        "depth": depth,
        "train ROC-AUC": roc_auc_score(y[fit_rows], tree.predict_proba(X.iloc[fit_rows])[:, 1]),
        "held-out ROC-AUC": roc_auc_score(y[check_rows],
                                          tree.predict_proba(X.iloc[check_rows])[:, 1]),
    })
curve = pd.DataFrame(curve).set_index("depth").round(4)
best_depth = int(curve["held-out ROC-AUC"].idxmax())
facts["tree_best_depth"] = (best_depth, float(curve.loc[best_depth, "held-out ROC-AUC"]))
facts["tree_depth20_train_vs_heldout"] = tuple(curve.loc[20])
display(curve)

fig, ax = plt.subplots(figsize=(7, 4))
curve.plot(ax=ax, marker="o")
ax.axvline(best_depth, color="grey", ls="--", lw=0.8)
ax.set(title="Decision tree: training vs held-out ROC-AUC by depth", xlabel="max depth",
       ylabel="ROC-AUC")
fig.tight_layout()
fig.savefig(FIG / "fig15_tree_depth_curve.png", dpi=150, bbox_inches="tight")
print("saved reports/figures/fig15_tree_depth_curve.png")

## The WoE scorecard: how a bank would score

One scorecard is fitted on the whole train split. The table shows the features with the highest
information value (IV), and then the points for the top 3 features. Convention: 600 points at
good:bad odds of 50:1, and every 20 points doubles the odds. An applicant's score is the sum of
the points of their bins; more points means safer.

In [ ]:
card = woe_scorecard(SEED).fit(X, y)
points = scorecard_points(card)
points.to_csv(PROJECT_ROOT / "reports" / "scorecard_points.csv", index=False)
iv = pd.Series(card.named_steps["woe"].iv_).sort_values(ascending=False)
facts["top_iv_features"] = iv.head(5).round(3).to_dict()
facts["scorecard_features_used"] = len(card.named_steps["woe"].selected_)
display(iv.head(15).round(3).rename("information value").to_frame())
for feature in card.named_steps["woe"].selected_[:3]:
    print(f"\n{feature}")
    rows = points[points["feature"] == feature][["bin", "woe", "points"]]
    print(rows.round(2).to_string(index=False))
print("\nsaved reports/scorecard_points.csv")

## Why Naive Bayes failed, and how PCA fixes it

Naive Bayes assumes every feature is independent of the others. Here dozens of columns copy each
other, so raw NB counts the same evidence many times. PCA turns the features into uncorrelated
components, which is much closer to what NB assumes.

In [ ]:
nb_raw = results["naive_bayes_raw"].summary["roc_auc"]
nb_pca = results["naive_bayes_pca"].summary["roc_auc"]
facts["naive_bayes_raw_vs_pca"] = (round(nb_raw, 4), round(nb_pca, 4))
print(f"Naive Bayes on raw features: {nb_raw:.4f} | on 30 PCA components: {nb_pca:.4f}")

## Random forest: the free out-of-bag check (optional, about 1–3 minutes)

Each tree only sees a random part of the data, so every applicant can be scored by the trees that
never saw them. This *out-of-bag* score is a built-in validation and should be close to the CV score.

In [ ]:
forest = random_forest(groups, SEED, oob=True).fit(X, y)
oob_auc = roc_auc_score(y, forest.named_steps["model"].oob_decision_function_[:, 1])
cv_auc = results["random_forest"].summary["roc_auc"]
facts["random_forest_oob_vs_cv"] = (round(oob_auc, 4), round(cv_auc, 4))
print(f"Random forest: out-of-bag ROC-AUC {oob_auc:.4f} vs 5-fold CV ROC-AUC {cv_auc:.4f}")

## Key facts
Send these to Claude to get the filled-in Findings.

In [ ]:
for key, value in facts.items():
    print(f"{key:<30} {value}")

## Findings  ✍️ (fill in after the run, or ask Claude to fill them from the Key facts)

- Every model was trained on the same 5 folds of the train split (184,503 applicants, 240 features).
- **Best model:** __ with ROC-AUC __ (95% CI __ to __). Runner-up: __ (__). The gap is / is not significant.
- **Boosting family:** LightGBM v1 __, XGBoost __, HistGradientBoosting __, AdaBoost __.
- **Other ensembles:** random forest __ (out-of-bag __ vs CV __); EBM (glass-box) __.
- **Linear models:** logistic regression __; WoE scorecard __ (__ features). The price of full transparency is __ ROC-AUC vs the best model.
- **SVMs:** linear __, RBF (Nystroem) __. **KNN** on 20 PCA components: __.
- **Naive Bayes:** raw __ vs on PCA components __, because PCA removes the correlations NB cannot handle.
- **Decision tree:** depth 8 __ in CV. The depth curve peaks at depth __ (held-out __); at depth 20 the training score is __ but held-out only __ (overfitting).
- **Highest-IV features:** __.

**Next (Step 7):** two experiments (class imbalance, tuning) and the final tuned model.